# 01. 전처리 — PDF → v5 표준화 parquet

## 이 노트북이 하는 일
1. 한국 LEED 건물 460개 PDF 스코어카드 로드
2. Rule 기반으로 v5 스키마로 표준화 (107개 매핑 규칙)
3. (선택) LLM 전문가 리뷰 — OPENAI_API_KEY 있을 때만
4. parquet 저장

## 출력
- `data/project_features.parquet` (460행 × 28컬럼)
- `data/standardized_credits.parquet` (9,747행)
- (LLM 옵션) `data/project_features_option_a.parquet`

## 참고
- `docs/01_전처리_과정.md`
- `docs/02_파이프라인_및_분석.md`

## Setup

In [1]:
import os
import sys
from pathlib import Path

# 노트북은 notebooks/ 에서 실행 → repo 루트로 cwd 이동
NOTEBOOK_DIR = Path.cwd()
ROOT = NOTEBOOK_DIR.parent
os.chdir(ROOT)
# src 는 notebooks/src/ 에 있음
sys.path.insert(0, str(NOTEBOOK_DIR))

try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / '.env')
except ImportError:
    pass

import pandas as pd
from src.langgraph_workflow.graph import run_standardization
from src.data.loader import LEEDDataLoader

print(f'작업 경로: {ROOT}')
print(f'OpenAI API 키: {"있음" if os.environ.get("OPENAI_API_KEY") else "없음 (Rule only)"}')

[RubricLoader] LEED_v4_for_Building_Design_and_Construction__1_PAGE.xlsx (v4) → 카테고리 8개: {'IP': 1.0, 'LT': 16.0, 'IEQ': 16.0, 'SS': 10.0, 'WE': 11.0, 'IN': 6.0, 'RP': 4.0, 'EA': 33.0}
[RubricLoader] LEED_v4_for_Building_Design_and_Construction__1_PAGE.xlsx (v4) → 카테고리 8개: {'IP': 1.0, 'LT': 16.0, 'IEQ': 16.0, 'SS': 10.0, 'WE': 11.0, 'IN': 6.0, 'RP': 4.0, 'EA': 33.0}
[RubricLoader] LEED_v4_for_Building_Design_and_Construction__1_PAGE.xlsx (v4) → 카테고리 8개: {'IP': 1.0, 'LT': 16.0, 'IEQ': 16.0, 'SS': 10.0, 'WE': 11.0, 'IN': 6.0, 'RP': 4.0, 'EA': 33.0}
[RubricLoader] LEED_v4_for_Building_Design_and_Construction__1_PAGE.xlsx (v4) → 카테고리 8개: {'IP': 1.0, 'LT': 16.0, 'IEQ': 16.0, 'SS': 10.0, 'WE': 11.0, 'IN': 6.0, 'RP': 4.0, 'EA': 33.0}
[RubricLoader] LEED_v4_for_Building_Design_and_Construction__1_PAGE.xlsx (v4) → 카테고리 8개: {'IP': 1.0, 'LT': 16.0, 'IEQ': 16.0, 'SS': 10.0, 'WE': 11.0, 'IN': 6.0, 'RP': 4.0, 'EA': 33.0}


[RubricLoader] LEED_v4_for_Interior_Design_and_Construction_Checklist.xlsx (v4) → 카테고리 7개: {'IP': 2.0, 'LT': 18.0, 'WE': 12.0, 'EA': 38.0, 'IN': 6.0, 'RP': 4.0, 'MR': 13.0}
[RubricLoader] LEED_v4_for_Interior_Design_and_Construction_Checklist.xlsx (v4) → 카테고리 7개: {'IP': 2.0, 'LT': 18.0, 'WE': 12.0, 'EA': 38.0, 'IN': 6.0, 'RP': 4.0, 'MR': 13.0}
[RubricLoader] LEED_v4_for_Interior_Design_and_Construction_Checklist.xlsx (v4) → 카테고리 7개: {'IP': 2.0, 'LT': 18.0, 'WE': 12.0, 'EA': 38.0, 'IN': 6.0, 'RP': 4.0, 'MR': 13.0}


[RubricLoader] LEED v4 for Building Operations and Maintenance Checklist_1 PAGE.XLSX (v4) → 카테고리 7개: {'MR': nan, 'LT': 17.0, 'SS': 10.0, 'WE': 12.0, 'IN': 6.0, 'RP': 4.0, 'EA': 38.0}
[RubricLoader] LEED_v4.1_for_Interior_Design_and_Construction_Checklist_Updated_4.26.XLSX (v4.1) → 카테고리 7개: {'IP': 2.0, 'LT': 18.0, 'WE': 12.0, 'EA': 38.0, 'IN': 6.0, 'RP': 4.0, 'MR': 13.0}
[RubricLoader] LEED_O_M_v4_1___scorecard_190213.xlsx (v4.1) → 카테고리 7개: {'MR': nan, 'LT': 22.0, 'SS': 4.0, 'IEQ': 20.0, 'IN': 1.0, 'WE': 15.0, 'EA': 100.0}
[RubricLoader] LEED_v5_Scorecard_BDC_Core_and_Shell.xlsx (v5) → 카테고리 7개: {'IP': 7, 'LT': 15, 'SS': 11, 'WE': 8, 'EA': 27, 'MR': 21, 'EQ': 11}


[RubricLoader] LEED_v5_Scorecard_BDC_New_Construction.xlsx (v5) → 카테고리 7개: {'IP': 1, 'LT': 15, 'SS': 11, 'WE': 9, 'EA': 33, 'MR': 18, 'EQ': 13}
[RubricLoader] LEED_v5_Scorecard_IDC.xlsx (v5) → 카테고리 6개: {'IP': 1, 'LT': 14, 'WE': 10, 'EA': 31, 'MR': 26, 'EQ': 18}
[RubricLoader] LEED_v5_Scorecard_OM.xlsx (v5) → 카테고리 7개: {'IP': 2, 'LT': 8, 'SS': 2, 'WE': 15, 'EA': 34, 'MR': 13, 'EQ': 26}
[RubricLoader] 총 15개 루브릭 파일 로딩 완료
[MappingRules] 107개 규칙 로딩 완료
작업 경로: D:\14. Dev Project\03. LEEDGRAPH
OpenAI API 키: 없음 (Rule only)


## 1단계: 데이터 로드

In [2]:
csv_df = LEEDDataLoader().load_project_directory()
pdf_files = sorted((ROOT / 'data' / 'scorecards').glob('*.pdf'))
print(f'CSV 프로젝트: {len(csv_df)}개, PDF: {len(pdf_files)}개')

프로젝트 디렉토리 로딩 완료: 456개 프로젝트
CSV 프로젝트: 456개, PDF: 460개


## 2단계: 전체 파이프라인 실행

각 PDF: PDF 파싱 → CSV 매칭 → Rule 매핑 → 수학 검증 → (LLM 리뷰) → 저장

- API 키 없음: 약 5분
- API 키 있음: 약 8시간 (~$55)

In [3]:
def build_credit_rows(project_id, version, leed_system, credit_mappings):
    rows = []
    for cm in credit_mappings:
        rows.append({
            'project_id': project_id,
            'source_version': version,
            'leed_system': leed_system,
            'source_credit_name': cm.get('credit_name', ''),
            'v5_credit_code': cm.get('v5_code', 'UNKNOWN'),
            'v5_category': cm.get('v5_category'),
            'points_awarded': cm.get('awarded', 0),
            'points_possible': cm.get('possible', 0),
            'mapping_method': 'rule' if cm.get('matched') else 'unmatched',
        })
    return rows

feature_rows, credit_rows, log_rows = [], [], []
counters = {'success': 0, 'failed': 0}

for i, pdf in enumerate(pdf_files, 1):
    if i % 50 == 0 or i == 1 or i == len(pdf_files):
        print(f'[{i:3d}/{len(pdf_files)}] {pdf.name[:55]}')
    try:
        state = run_standardization(pdf_path=str(pdf), directory_df=csv_df)
        if state.get('status') != 'completed':
            raise RuntimeError(f"status={state.get('status')}")

        final = state['final_v5_data']
        project = state.get('project', {})
        rule_result = state.get('rule_mapping_result', {})
        math_result = state.get('math_validation_result', {})
        val_result = state.get('validation_result', {}) or {}

        version = final.get('original_version', 'unknown')
        project_id = final.get('project_id', pdf.name)
        leed_system = final.get('leed_system', '')

        feature_rows.append({
            'project_id': project_id,
            'project_name': final.get('project_name', ''),
            'leed_system': leed_system,
            'building_type': final.get('building_type', ''),
            'gross_area_sqm': final.get('gross_area_sqm', 0),
            'original_version': version,
            'certification_level': final.get('certification_level', ''),
            'total_score_original': final.get('total_score_original', 0),
            'total_score_v5': final.get('total_score_v5', 0),
            'achievement_ratio_original': final.get('achievement_ratio_original', 0),
            'achievement_ratio_v5': final.get('achievement_ratio_v5', 0),
            'standardization_track': final.get('standardization_track', 'rule'),
            'drift': math_result.get('ratio_drift', 0),
            'credit_rule_hit_rate': rule_result.get('credit_rule_hit_rate', None),
            **{k: v for k, v in final.items() if k.startswith('ratio_')},
            **{k: v for k, v in final.items() if k.startswith('score_v5_')},
        })

        cm = rule_result.get('credit_mappings', [])
        if cm:
            credit_rows.extend(build_credit_rows(project_id, version, leed_system, cm))
        else:
            for cat in rule_result.get('mapped_categories', {}):
                credit_rows.append({
                    'project_id': project_id, 'source_version': version, 'leed_system': leed_system,
                    'source_credit_name': f'[category] {cat}',
                    'v5_credit_code': f'CAT_{cat}', 'v5_category': cat,
                    'points_awarded': project.get('categories', {}).get(cat, 0),
                    'points_possible': project.get('categories_possible', {}).get(cat, 0),
                    'mapping_method': 'category_proportional',
                })

        if val_result:
            log_rows.append({
                'project_id': project_id,
                'llm_review_target': val_result.get('target'),
                'llm_review_is_valid': val_result.get('is_valid'),
                'llm_review_score': val_result.get('validation_score'),
                'llm_review_issues': '; '.join(val_result.get('issues', []))[:300],
                'llm_review_feedback': (val_result.get('feedback') or '')[:300],
            })

        counters['success'] += 1
    except Exception:
        counters['failed'] += 1

print(f'\n완료: {counters}')

[  1/460] Scorecard_adidasBrandFlagshipSeoul_230501.pdf


[ 50/460] Scorecard_BurberryLotteBusanWomen_220628.pdf


[100/460] Scorecard_ChristianDiorSeoulTheHyundai_221017.pdf


[150/460] Scorecard_FactorialSeongsu_240822.pdf


[200/460] Scorecard_IBKFINANCETOWER_161116.pdf


...

[350/460] Scorecard_PyeongtaekLogisticsPark_250314.pdf


[400/460] Scorecard_SongdoBlockD19_120531.pdf


[450/460] Scorecard_WestgateTower_240303.pdf


[460/460] Scorecard_ZaraKangnam_181127.pdf



완료: {'success': 460, 'failed': 0}


## 3단계: parquet 저장

In [4]:
DATA = ROOT / 'data'

feat_df = pd.DataFrame(feature_rows)
cred_df = pd.DataFrame(credit_rows)

feat_df.to_parquet(DATA / 'project_features.parquet', index=False)
cred_df.to_parquet(DATA / 'standardized_credits.parquet', index=False)

if log_rows:
    log_df = pd.DataFrame(log_rows)
    log_df['project_id'] = log_df['project_id'].astype(str)
    feat_df['project_id'] = feat_df['project_id'].astype(str)
    merged = feat_df.merge(log_df, on='project_id', how='left')
    merged['has_llm_review'] = merged['llm_review_target'].notna()
    merged.to_parquet(DATA / 'project_features_option_a.parquet', index=False)
    print(f'Option A parquet: {len(merged)}행 (LLM 리뷰 {merged["has_llm_review"].sum()}건)')

print(f'project_features.parquet: {len(feat_df)}행')
print(f'standardized_credits.parquet: {len(cred_df)}행')

project_features.parquet: 460행
standardized_credits.parquet: 9747행


## 4단계: 검증

In [5]:
print('=== 기본 통계 ===')
print(f'행수: {len(feat_df)}')
print(f"버전: {feat_df['original_version'].value_counts().to_dict()}")
print(f"등급: {feat_df['certification_level'].value_counts().to_dict()}")
print(f"평균 drift: {feat_df['drift'].mean()*100:.2f}%")
print(f"drift > 20%: {(feat_df['drift'] > 0.20).sum()}건")
print(f"\n크레딧 매핑 방식:\n{cred_df['mapping_method'].value_counts()}")

=== 기본 통계 ===
행수: 460
버전: {'v4': 276, 'v2009': 114, 'v4.1': 48, 'v2.2': 18, 'v2.0': 4}
등급: {'Gold': 235, 'Silver': 118, 'Platinum': 56, 'Certified': 51}
평균 drift: 11.89%
drift > 20%: 47건

크레딧 매핑 방식:
mapping_method
rule                     7697
unmatched                1127
category_proportional     923
Name: count, dtype: int64


## 다음

`02_데이터분석.ipynb` 에서 XGBoost + SHAP 실행.